In [ ]:
%load ../lib/RecursiveSet.jsh
%load Formula.jsh

The file `Formula.jsh` defines the interface `Literal` and the record classes `Var` and `Neg`: A propositional variable $p$ is represented as `new Var("p")`, while the negated variable $\neg p$ is represented as `new Neg("p")`.  A clause is a set of literals, i.e. a `RecursiveSet<Literal>`.

In [ ]:
@SafeVarargs
<T> RecursiveSet<T> set(T... elements) {
    return new RecursiveSet<>(elements);
}

In order to write down clauses succinctly, we define the variables `p`, `q`, `r`, and `s`.  Furthermore, the function `neg` takes a propositional variable $v$ and returns the negative literal $\neg v$.

In [ ]:
var p = new Var("p");
var q = new Var("q");
var r = new Var("r");
var s = new Var("s");

In [ ]:
Literal neg(Var v) {
    return new Neg(v.name());
}

# Refutational Completeness of the Cut Rule

This notebook implements a number of procedures that are needed in our proof of the <em style="color:blue">refutational completeness</em> of the cut rule.

The function $\texttt{complement}(l)$ computes the *complement* of a literal $l$.
As we are only dealing with formulas in CNF that are written in set notation and these formulas do not contain $\top$ or $\bot$, a literal is either 
a propositional variable or the negation of a propositional variable.
Therefore, if $p$ is a propositional variable, we have the following: 
* $\texttt{complement}(p) = \neg p$,
* $\texttt{complement}(\neg p) = p$.

In [ ]:
Literal complement(Literal l) {
    return switch (l) {
        case Var(var x) -> new Neg(x);
        case Neg(var x) -> new Var(x);
        default -> throw new IllegalArgumentException("unknown literal: " + l);
    };
}

In [ ]:
complement(p)

In [ ]:
complement(neg(p))

The function $\texttt{extractVariable}(l)$ extracts the propositional variable from the literal $l$.
If $p$ is a propositional variable, we have the following: 

1. $\texttt{extractVariable}(p) = p$,
2. $\texttt{extractVariable}(\neg p) = p$.

In [ ]:
String extractVariable(Literal l) {
    return switch (l) {
        case Var(var x) -> x;
        case Neg(var x) -> x;
        default -> throw new IllegalArgumentException("unknown literal: " + l);
    };
}

In [ ]:
extractVariable(p)

In [ ]:
extractVariable(neg(p))

The function $\texttt{collectsVariables}(M)$ takes a set of clauses $M$ as its input and computes the set of all propositional variables occurring in $M$.  The clauses in $M$ are represented as sets of literals.

In [ ]:
RecursiveSet<String> collectVariables(RecursiveSet<RecursiveSet<Literal>> M) {
    return flatMap(M, C -> C.map(l -> extractVariable(l)));
}

In [ ]:
RecursiveSet<Literal> C1 = set(p, q, r);
RecursiveSet<Literal> C2 = set(neg(p), neg(q), neg(s));
var M = set(C1, C2);
collectVariables(M)

As literals are printed as `p` and `¬p` and sets are printed using curly braces, we do not need a function to convert clauses into strings.

In [ ]:
M

Given two clauses $C_1$ and $C_2$ that are represented as sets of literals, the function `cutRule`$(C_1, C_2)$ computes the set of all clauses that can be derived from $C_1$ and $C_2$ using the *cut rule*.  In set notation, the cut rule is the following rule of inference:
$$
   \frac{\displaystyle \;C_1\cup \{l\} \quad C_2 \cup \bigl\{\overline{\,l\,}\bigr\}}{\displaystyle C_1 \cup C_2}
$$

In [ ]:
RecursiveSet<RecursiveSet<Literal>> cutRule(RecursiveSet<Literal> C1, RecursiveSet<Literal> C2) {
    return C1.filterMap(
        l -> C2.contains(complement(l)),
        l -> {
            var lBar = complement(l);
            // (C1 \ {l}) ∪ (C2 \ {lBar})
            return C1.difference(set(l)).union(C2.difference(set(lBar)));
        }
    );
}

In [ ]:
RecursiveSet<Literal> C1 = set(neg(q), neg(r));
RecursiveSet<Literal> C2 = set(q, r);
cutRule(C1, C2)

In the expression `saturate(Clauses)` below, `Clauses` is a set of *clauses*, where each clause is a set of *literals*.  The call `saturate(Clauses)` computes the set of all clauses that can be derived from clauses in the set `Clauses` using the *cut rule*.  The function keeps applying the cut rule until either no new clauses can be derived, or the empty clause $\{\}$ is derived.  The resulting set of clauses is *saturated* in the following sense:  If $C_1$ and $C_2$ are clauses from the set `Clauses` and the clause $D$ can be derived from $C_1$ and $C_2$ via the cut rule, then $D \in \texttt{Clauses}$ or $\{\} \in \texttt{Clauses}$.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> saturate(RecursiveSet<RecursiveSet<Literal>> Clauses) {
    var currentClauses = Clauses;
    while (true) {
        int sizeBefore = currentClauses.size();
        // 1. Generate all possible pairs of clauses (C1, C2)
        var pairs = currentClauses.cartesianProduct(currentClauses);
        // 2. Map each pair through the cut rule, flattening the results into one set
        var derived = flatMap(pairs, pair -> cutRule(pair.first(), pair.second()));
        if (derived.some(D -> D.isEmpty())) {
            return set(set());
        }
        currentClauses = currentClauses.union(derived);
        if (currentClauses.size() == sizeBefore) {
            return currentClauses;
        }
    }
}

In [ ]:
RecursiveSet<Literal> C1 = set(p, q);
RecursiveSet<Literal> C2 = set(neg(p));
RecursiveSet<Literal> C3 = set(neg(p), neg(q));
var Clauses = set(C1, C2, C3);
saturate(Clauses)

The function $\texttt{findValuation}(\texttt{Clauses})$ takes a set of clauses as input.  The function tries to compute a variable interpretation that makes all of these clauses `true`.  If this attempt is successful, a set of literals is returned.  This set of literals does not contain any complementary literals and therefore corresponds to a variable assignment satisfying all clauses.  The result is wrapped into an `Optional`.  If $\texttt{Clauses}$ is unsatisfiable, `Optional.empty()` is returned.

In [ ]:
Optional<RecursiveSet<Literal>> findValuation(RecursiveSet<RecursiveSet<Literal>> Clauses) {
    var Saturated = saturate(Clauses);
    if (Saturated.some(C -> C.isEmpty())) {
        return Optional.empty();
    }
    var Variables = collectVariables(Clauses);
    RecursiveSet<Literal> Literals = set();
    for (var x : Variables) {
        var v = new Var(x);
        // A clause forces v to be true if it contains v
        // AND all OTHER literals in the clause evaluate to false.
        // (i.e., their complements are already in our set Literals)
        boolean forcesV = Saturated.some(
            C -> C.contains(v) &&
                 C.difference(set(v)).every(l -> Literals.contains(complement(l)))
        );
        Literals.add(forcesV ? v : complement(v));
    }
    return Optional.of(Literals);
}

In [ ]:
RecursiveSet<Literal> C01 = set(r, p, s);
RecursiveSet<Literal> C02 = set(r, s);
RecursiveSet<Literal> C03 = set(p, q, s);
RecursiveSet<Literal> C04 = set(neg(p), neg(q));
RecursiveSet<Literal> C05 = set(neg(p), s, neg(r));
RecursiveSet<Literal> C06 = set(p, neg(q), r);
RecursiveSet<Literal> C07 = set(neg(r), neg(s), q);
RecursiveSet<Literal> C08 = set(neg(p), neg(s));
RecursiveSet<Literal> C09 = set(p, neg(r), neg(q));
RecursiveSet<Literal> C10 = set(neg(p), r, q, neg(s));
RecursiveSet<Literal> C11 = set(p, q, r, neg(s));

var Clauses1 = set(C01, C02, C03, C04, C05, C06, C07, C08, C09, C10);

var valuation = findValuation(Clauses1);
if (valuation.isPresent()) {
    System.out.println(valuation.get());
} else {
    System.out.println("unsatisfiable");
}

In [ ]:
var Clauses2 = set(C01, C02, C03, C04, C05, C06, C07, C08, C09, C10, C11);
var valuation = findValuation(Clauses2);
if (valuation.isPresent()) {
    System.out.println(valuation.get());
} else {
    System.out.println("unsatisfiable");
}

In [ ]:
saturate(Clauses2)